# NovaMind RAG Evaluation Harness

This notebook evaluates a RAG pipeline across different configurations using RAGAS metrics.

**Pipeline stages:**
1. **Ingest** — split corpus into chunks, embed with BGE-M3, store in FAISS
2. **Retrieve** — MMR search, optional hybrid (BM25 + dense), optional reranking
3. **Generate** — answer questions using the retrieved context via Ollama
4. **Evaluate** — score answers with RAGAS (faithfulness, recall, precision, correctness)

**Required servers (must be running before executing cells):**
- Ollama at `OPENAI_BASE_URL` — generation + RAGAS judge
- BGE-M3 embedding server at `EMBEDDINGS_BASE_URL`
- *(optional)* BGE-Reranker at `RERANK_BASE_URL`

**Required files:**
- `eval_dataset.csv` — columns: `question`, `ground_truth`
- *(optional)* a `.pdf` or `.txt` corpus file — set `CORPUS_FILE` in Cell 2, or leave `None` to use the built-in ML corpus

---
## Cell 1 — Imports & Setup

Imports all libraries and suppresses noisy LangChain/HuggingFace logs.
Also patches `print` for safe Unicode output on Windows.

In [5]:
# !pip install ragas python-dotenv --quiet

In [6]:
import os, sys, json, csv, time, datetime, textwrap, re
from pathlib import Path
from typing import Optional
from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from dotenv import load_dotenv
load_dotenv()  # must be first

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

from ragas import evaluate
from ragas.metrics import Faithfulness, ContextRecall, ContextPrecision, AnswerCorrectness
faithfulness       = Faithfulness()
context_recall     = ContextRecall()
context_precision  = ContextPrecision()
answer_correctness = AnswerCorrectness()
from datasets import Dataset as HFDataset

import logging
logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)

os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_TRACING"]    = "false"

def safe_print(*args, **kwargs):
    sep, end, flush = kwargs.get("sep"," "), kwargs.get("end","\n"), kwargs.get("flush",False)
    text = sep.join(str(a) for a in args) + end
    try:
        sys.stdout.write(text)
    except UnicodeEncodeError:
        enc = getattr(sys.stdout, "encoding", None) or "utf-8"
        sys.stdout.buffer.write(text.encode(enc, errors="replace"))
    if flush:
        sys.stdout.flush()

print = safe_print
print("✅ Imports OK")

✅ Imports OK


C:\Users\Taktouk\AppData\Local\Temp\ipykernel_7208\624278670.py:18: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ContextRecall, ContextPrecision, AnswerCorrectness
C:\Users\Taktouk\AppData\Local\Temp\ipykernel_7208\624278670.py:18: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import Faithfulness, ContextRecall, ContextPrecision, AnswerCorrectness
C:\Users\Taktouk\AppData\Local\Temp\ipykernel_7208\624278670.py:18: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas

---
## Cell 2 — Configuration

All tunable settings live here. Values are read from `.env` first; the defaults shown are used as fallback.

| Variable | What it controls |
|---|---|
| `LLM_MODEL` | Model served by Ollama for generation AND RAGAS scoring |
| `LLM_BASE_URL` | Ollama OpenAI-compatible endpoint |
| `EMBEDDINGS_BASE_URL` | Remote BGE-M3 embedding server |
| `EVAL_CSV` | Path to your evaluation question/answer CSV |
| `CORPUS_FILE` | Path to `.pdf` / `.txt` to index — `None` uses built-in ML corpus |
| `EVAL_WORKERS` | Parallel threads for RAG generation (keep ≤ Ollama queue depth) |
| `RAGAS_CONCURRENCY` | Parallel async jobs inside RAGAS scoring (keep `1` for Ollama) |
| `RAGAS_TIMEOUT` | Seconds before a single RAGAS LLM call is abandoned |

In [7]:
# ── API keys ──────────────────────────────────────────────────────────────────
MISTRAL_API_KEY    = os.getenv("MISTRAL_API_KEY", "")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")

## ── Generation LLM (Ollama) ───────────────────────────────────────────────────
# LLM_MODEL       = os.getenv("LLM_MODEL",      "gemma4:26b")
# LLM_BASE_URL    = os.getenv("OPENAI_BASE_URL", "http://192.168.130.177:11434/v1")
# LLM_API_KEY     = os.getenv("OPENAI_API_KEY",  "not-needed")
# LLM_TEMPERATURE = float(os.getenv("LLM_TEMPERATURE", "0.1"))
# LLM_MAX_TOKENS  = int(os.getenv("LLM_MAX_TOKENS",    "2048"))

LLM_MODEL = "mistral-small-latest"
# # ── Embeddings (remote BGE-M3) ────────────────────────────────────────────────
# EMBEDDINGS_BASE_URL = os.getenv("EMBEDDINGS_BASE_URL",   "http://192.168.130.177:8081/v1")
# EMBEDDINGS_MODEL    = os.getenv("EMBEDDINGS_MODEL",      "BAAI/bge-m3")
# EMBEDDINGS_API_KEY  = os.getenv("EMBEDDINGS_API_KEY",    "not-needed")
# EMBEDDINGS_DIMS     = int(os.getenv("EMBEDDINGS_DIMENSIONS", "1024"))  # NOTE: unused
EMBEDDINGS_MODEL = "BAAI/bge-base-en-v1.5"

# # ── Reranker ──────────────────────────────────────────────────────────────────
# RERANK_BASE_URL = os.getenv("RERANK_BASE_URL", "http://192.168.130.177:8082/v1")
# RERANK_MODEL    = os.getenv("RERANK_MODEL",    "BAAI/bge-reranker-v2-m3")
# RERANK_API_KEY  = os.getenv("RERANK_API_KEY",  "not-needed")

EMBED_MODEL = EMBEDDINGS_MODEL
JUDGE_MODEL = LLM_MODEL
GEN_MODEL   = LLM_MODEL

# ── Paths ─────────────────────────────────────────────────────────────────────
EVAL_CSV    = os.getenv("EVAL_CSV",    "eval_dataset.csv")
CORPUS_FILE = os.getenv("CORPUS_FILE", None)
OUT_DIR     = os.getenv("OUT_DIR",     ".")

# ── Performance ───────────────────────────────────────────────────────────────
EVAL_WORKERS       = int(os.getenv("EVAL_WORKERS",       "4"))
RAGAS_CONCURRENCY = int(os.getenv("RAGAS_CONCURRENCY", "2"))  
RAGAS_TIMEOUT     = int(os.getenv("RAGAS_TIMEOUT",     "60")) 
RAGAS_EVAL_SAMPLES = int(os.getenv("RAGAS_EVAL_SAMPLES", "10"))

# print(f"LLM  : {LLM_MODEL} @ {LLM_BASE_URL}")
# print(f"Embed: {EMBEDDINGS_MODEL} @ {EMBEDDINGS_BASE_URL}")
print(f"LLM  : {LLM_MODEL}")
print(f"Embed: {EMBEDDINGS_MODEL}")
print(f"CSV  : {EVAL_CSV}")
print(f"Corpus: {CORPUS_FILE or 'built-in ML corpus'}")

LLM  : mistral-small-latest
Embed: BAAI/bge-base-en-v1.5
CSV  : eval_dataset.csv
Corpus: built-in ML corpus


---
## Cell 3 — Built-in Corpus

A short ML reference text used when no external document is provided.
If `CORPUS_FILE` is set in Cell 2, this cell is still defined but never used.

In [ ]:
BUILTIN_CORPUS = """
Machine Learning is a subset of artificial intelligence that allows systems to learn
and improve from experience without being explicitly programmed. It focuses on developing
computer programs that can access data and use it to learn for themselves.

Supervised Learning is a type of machine learning where the algorithm learns from labeled
training data. The algorithm learns to map input features to output labels. Examples include
linear regression for predicting continuous values and logistic regression for classification.

Gradient Descent is an optimization algorithm used to minimize the cost function in machine
learning. It works by iteratively moving in the direction of steepest descent as defined by
the negative of the gradient. The learning rate controls how large each step is.

Overfitting occurs when a model learns the training data too well, capturing noise and
random fluctuations. This leads to poor generalization to new unseen data. Regularization
techniques like L1 (Lasso) and L2 (Ridge) help prevent overfitting by adding penalty terms
to the loss function.

Neural Networks are computational models inspired by the human brain. They consist of
interconnected layers of nodes called neurons. Backpropagation is the key algorithm for
training neural networks, using the chain rule to compute gradients and update weights.
The vanishing gradient problem affects deep networks when gradients become too small for
early layers to learn effectively.

Support Vector Machines (SVM) find the optimal hyperplane that maximally separates classes.
The margin is the distance between the hyperplane and the nearest data points (support vectors).
Kernel functions allow SVMs to handle non-linearly separable data.

Decision Trees split data recursively based on feature thresholds. Random Forests are an
ensemble of decision trees using bagging. Gradient Boosting builds trees sequentially, each
correcting the errors of the previous one.

Cross-validation assesses model generalization by training on subsets and testing on held-out
data. k-fold cross-validation splits data into k equal parts and rotates the test fold.

Principal Component Analysis (PCA) is a dimensionality reduction technique that finds
orthogonal axes of maximum variance. It projects data onto fewer dimensions while preserving
as much variance as possible.

The bias-variance tradeoff: high bias means underfitting (model too simple), high variance
means overfitting (model too complex). The goal is to find the sweet spot that minimizes total error.

Precision = TP / (TP + FP). Recall = TP / (TP + FN). F1 score is the harmonic mean of precision
and recall. The confusion matrix summarizes classification results across all classes.

K-Nearest Neighbors (KNN) classifies new points by majority vote of the k closest training
examples in feature space. The choice of k and distance metric heavily influence performance.
"""


CORPUS_FILE = os.getenv("CORPUS_FILE", ".\LGLSI-3-25-103.pdf")
# if CORPUS_FILE and Path(CORPUS_FILE).is_file():
#     corpus_text = load_corpus(CORPUS_FILE)
#     print(f"Loaded corpus from {CORPUS_FILE}: {len(corpus_text):,} characters")
# else:
#     corpus_text = BUILTIN_CORPUS
#     print(f"Using built-in ML corpus ({len(corpus_text):,} characters)")

---
## Cell 4 — Singleton Client Cache

Creating a new `ChatOpenAI` or `OpenAIEmbeddings` client on every call adds connection overhead.
These functions create each client **once** and reuse it for the entire notebook session.
The RAGAS judge/embeddings also cache their smoke-test result so it only runs on first call.

In [9]:
_llm_client_cache:   dict = {}
_pipeline_embeddings = None
_ragas_judge         = None
_ragas_embeddings    = None
_cross_encoder       = None


def get_llm_client(temperature: float = 0.1, max_tokens: int = 1024):
    """Cached ChatMistralAI — one instance per (temperature, max_tokens) pair."""
    global _llm_client_cache
    key = (temperature, max_tokens)
    if key not in _llm_client_cache:
        from langchain_mistralai import ChatMistralAI
        _llm_client_cache[key] = ChatMistralAI(
            model="mistral-small-latest",
            mistral_api_key=MISTRAL_API_KEY,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        print(f"  [gen] mistral-small-latest (temp={temperature}, max_tokens={max_tokens}) ready")
    return _llm_client_cache[key]


def get_pipeline_embeddings():
    global _pipeline_embeddings
    if _pipeline_embeddings is None:
        from langchain_community.embeddings import HuggingFaceEmbeddings
        _pipeline_embeddings = HuggingFaceEmbeddings(model_name=EMBEDDINGS_MODEL)
        print(f"  [embed] HuggingFace {EMBEDDINGS_MODEL} ready")
    return _pipeline_embeddings


def get_cross_encoder():
    global _cross_encoder
    if _cross_encoder is None:
        from sentence_transformers import CrossEncoder
        _cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    return _cross_encoder


def get_ragas_judge():
    global _ragas_judge
    if _ragas_judge is not None:
        return _ragas_judge if _ragas_judge is not False else None
    if MISTRAL_API_KEY:
        try:
            from langchain_mistralai import ChatMistralAI
            from ragas.llms import LangchainLLMWrapper
            raw = ChatMistralAI(model="mistral-small-latest",
                                mistral_api_key=MISTRAL_API_KEY, temperature=0)
            raw.invoke("Reply with one word: ok")
            wrapped = LangchainLLMWrapper(raw)
            print("  [judge] mistral-small-latest ready")
            _ragas_judge = wrapped
            return wrapped
        except Exception as e:
            print(f"  [judge] Mistral failed ({e})")
    print("  [judge] WARNING: No judge LLM available — LLM metrics will be n/a")
    _ragas_judge = False
    return None


def get_ragas_embeddings():
    global _ragas_embeddings
    if _ragas_embeddings is not None:
        return _ragas_embeddings if _ragas_embeddings is not False else None
    try:
        from ragas.embeddings import LangchainEmbeddingsWrapper
        wrapped = LangchainEmbeddingsWrapper(get_pipeline_embeddings())
        print(f"  [embed] HuggingFace {EMBEDDINGS_MODEL} ready")
        _ragas_embeddings = wrapped
        return wrapped
    except Exception as e:
        print(f"  [embed] embeddings failed ({e})")
        _ragas_embeddings = False
        return None


print("✅ Singleton cache functions defined")


✅ Singleton cache functions defined


---
## Cell 5 — Generation Helpers

`generate_with_openai` sends a prompt to Ollama and returns the text response.
`generate_answer` is the top-level wrapper that calls it and returns `(answer, model_name)`.

In [10]:
def generate_with_openai(system_prompt: str, question: str,
                         temperature: float = 0.1, max_tokens: int = 1024) -> str:
    llm = get_llm_client(temperature=temperature, max_tokens=max_tokens)
    return llm.invoke(f"{system_prompt}\n\nQuestion: {question}\n\nAnswer:").content.strip()


def generate_answer(system_prompt: str, question: str,
                    temperature: float = 0.1, max_tokens: int = 1024):
    """Returns (answer: str, model_name: str). Falls back to error string on failure."""
    try:
        answer = generate_with_openai(system_prompt, question,
                                      temperature=temperature, max_tokens=max_tokens)
        return answer, os.getenv("LLM_MODEL")
    except Exception as e:
        print(f"  [gen] call failed ({e})")
    return "ERROR: No LLM available for generation.", ""


print("✅ Generation helpers defined")


✅ Generation helpers defined


---
## Cell 6 — Rerankers

Two optional reranking strategies that can be switched on per-experiment:

- **Semantic reranker** — re-scores retrieved chunks by cosine similarity to the query using the same BGE-M3 embeddings
- **Cross-encoder reranker** — uses a dedicated `ms-marco-MiniLM` cross-encoder model for more accurate (but slower) relevance scoring

In [11]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def rerank_contexts(query, contexts, embedder, top_k=6):
    """Semantic reranker: re-score chunks by cosine similarity with the query embedding."""
    query_emb    = embedder.embed_query(query)
    context_embs = embedder.embed_documents(contexts)
    scored = [(ctx, cosine_similarity(query_emb, emb)) for ctx, emb in zip(contexts, context_embs)]
    return [ctx for ctx, _ in sorted(scored, key=lambda x: x[1], reverse=True)[:top_k]]


def rerank_cross_encoder(query, contexts, reranker, top_k=6):
    """Cross-encoder reranker: score (query, chunk) pairs with ms-marco-MiniLM."""
    pairs  = [[query, ctx] for ctx in contexts]
    scores = reranker.predict(pairs)
    return [ctx for ctx, _ in sorted(zip(contexts, scores), key=lambda x: x[1], reverse=True)[:top_k]]


print("✅ Rerankers defined")

✅ Rerankers defined


---
## Cell 7 — RAG Pipeline

The `RAGPipeline` class encapsulates the full ingest → retrieve → generate flow.

**Key parameters:**

| Parameter | Default | Effect |
|---|---|---|
| `chunk_size` | 1000 | Characters per chunk |
| `chunk_overlap` | 200 | Overlap between adjacent chunks |
| `top_k` | 6 | Chunks returned per query |
| `use_reformulation` | False | Rewrite query via LLM before retrieval |
| `use_reranker` | False | Semantic cosine reranking |
| `use_cross_encoder` | False | Cross-encoder reranking (slower, more accurate) |
| `use_multiquery` | False | Generate 3 query variants and merge results |
| `use_postprocessing` | False | Filter answer sentences by keyword overlap |
| `use_parent_doc` | False | Expand retrieved chunks to neighbouring context window |
| `use_hybrid` | False | Merge dense (FAISS MMR) + sparse (BM25) results |

In [12]:
class RAGPipeline:
    def __init__(
        self,
        chunk_size=1000, chunk_overlap=200, top_k=6,
        use_reformulation=False, embed_model: str = EMBED_MODEL,
        use_reranker=False, use_cross_encoder=False, use_multiquery=False,
        use_postprocessing=False, use_parent_doc=False, use_hybrid=False,
        use_llm_reranker=False,
        fetch_k: int = 20,
        lambda_mult: float = 0.7,
        temperature: float = 0.1,
        max_tokens: int = 1024,
        similarity_threshold: float = 0.0,
        n_queries: int = 3,
    ):
        self.chunk_size           = chunk_size
        self.chunk_overlap        = chunk_overlap
        self.top_k                = top_k
        self.use_reformulation    = use_reformulation
        self.use_reranker         = use_reranker
        self.use_cross_encoder    = use_cross_encoder
        self.use_multiquery       = use_multiquery
        self.use_postprocessing   = use_postprocessing
        self.use_parent_doc       = use_parent_doc
        self.use_hybrid           = use_hybrid
        self.use_llm_reranker     = use_llm_reranker
        self.fetch_k              = fetch_k
        self.lambda_mult          = lambda_mult
        self.temperature          = temperature
        self.max_tokens           = max_tokens
        self.similarity_threshold = similarity_threshold
        self.n_queries            = n_queries
        self.embeddings           = get_pipeline_embeddings()
        self.vectorstore          = None
        if self.use_cross_encoder:
            self.reranker = get_cross_encoder()

    def generate_queries(self, question):
        if not self.use_multiquery:
            return [question]
        variants = [question, f"Explain: {question}", f"Describe: {question}",
                    f"What is meant by: {question}", f"Summarize: {question}"]
        return variants[:self.n_queries]

    def ingest_text(self, text: str, source: str = "corpus") -> int:
        from langchain_community.retrievers import BM25Retriever
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""],
        )
        raw_chunks = splitter.create_documents([text], metadatas=[{"source": source}])
        chunks = []
        for i, chunk in enumerate(raw_chunks):
            if self.use_parent_doc:
                chunk.metadata["parent_id"]   = i // 3
                start = max(0, i - 1); end = min(len(raw_chunks), i + 1)
                chunk.metadata["parent_text"] = " ".join(
                    raw_chunks[j].page_content for j in range(start, end))
            chunks.append(chunk)
        self.vectorstore = FAISS.from_documents(chunks, self.embeddings)
        if self.use_hybrid:
            self.bm25   = BM25Retriever.from_documents(chunks)
            self.bm25.k = self.top_k
        print(f"  indexed {len(chunks)} chunks from '{source}'")
        return len(chunks)

    def ingest_file(self, path: str) -> int:
        p = Path(path)
        if p.suffix == ".pdf":
            from langchain_community.document_loaders import PyPDFLoader
            text = "\n\n".join(d.page_content for d in PyPDFLoader(str(p)).load())
        else:
            text = p.read_text(encoding="utf-8")
        return self.ingest_text(text, source=p.name)

    def _reformulate(self, question: str) -> str:
        prompt = textwrap.dedent(f"""
            You are a search query optimizer for a RAG system.
            Rules:
            - Keep proper nouns, company names, technical terms EXACTLY as written
            - Expand only generic words with synonyms
            - Return ONLY the reformulated query, nothing else
            Original: {question}
            Reformulated:
        """).strip()
        try:
            return generate_answer(prompt, question,
                                   temperature=self.temperature, max_tokens=64)[0]
        except Exception:
            return question

    def retrieve(self, question: str) -> tuple:
        if self.vectorstore is None:
            raise RuntimeError("Call ingest_text() first.")
        effective = self._reformulate(question) if self.use_reformulation else question
        all_docs  = []
        for q in self.generate_queries(question):
            dense_docs = self.vectorstore.max_marginal_relevance_search(
                q, k=self.top_k, fetch_k=self.fetch_k, lambda_mult=self.lambda_mult)
            if self.use_hybrid:
                sparse_docs = self.bm25.invoke(q)
                seen_t, merged = set(), []
                for d in dense_docs + sparse_docs:
                    if d.page_content not in seen_t:
                        seen_t.add(d.page_content); merged.append(d)
                all_docs.extend(merged)
            else:
                all_docs.extend(dense_docs)
        seen = {}
        for d in all_docs:
            key = d.metadata.get("parent_id") if self.use_parent_doc else d.page_content
            if key not in seen: seen[key] = d
        candidates = list(seen.values())
        raw_texts  = [d.page_content for d in candidates]

        # similarity threshold filter
        if self.similarity_threshold > 0.0:
            query_emb = self.embeddings.embed_query(effective)
            doc_embs  = self.embeddings.embed_documents(raw_texts)
            def _cos(a, b):
                return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))
            filtered = [(txt, doc) for txt, emb, doc in zip(raw_texts, doc_embs, candidates)
                        if _cos(query_emb, emb) >= self.similarity_threshold]
            if filtered:
                raw_texts  = [t for t, _ in filtered]
                candidates = [d for _, d in filtered]

        if self.use_cross_encoder:
            raw_texts = rerank_cross_encoder(effective, raw_texts, self.reranker, top_k=self.top_k)
            contexts  = [d.metadata.get("parent_text", d.page_content)
                         for d in candidates if d.page_content in raw_texts]
        elif self.use_reranker:
            raw_texts = rerank_contexts(effective, raw_texts, self.embeddings, top_k=self.top_k)
            contexts  = [d.metadata.get("parent_text", d.page_content)
                         for d in candidates if d.page_content in raw_texts]
        elif self.use_llm_reranker:
            raw_texts = self.rerank_llm(effective, raw_texts, top_k=self.top_k)
            contexts  = raw_texts
        else:
            contexts  = [d.metadata.get("parent_text", d.page_content)
                         if self.use_parent_doc else d.page_content
                         for d in candidates[:self.top_k]]
        return effective, contexts

    def overlap_score(self, q, s):
        q_w = set(q.lower().split()); s_w = set(s.lower().split())
        return len(q_w & s_w) / len(s_w) if s_w else 0

    def postprocess(self, answer, question):
        if not self.use_postprocessing:
            return answer
        sentences = answer.split(".")
        return ". ".join(s for s in sentences if self.overlap_score(question, s) >= 0.1) + "."

    def generate(self, question: str, contexts: list) -> str:
        context_block = "\n\n---\n\n".join(contexts)
        system_prompt = textwrap.dedent(f"""You are a helpful assistant.
Answer the question based on the provided context.
Synthesize and summarize relevant information even if it's spread across multiple parts.
Be clear and concise.
If the context contains no relevant information at all, say: \"I don't have enough information to answer this.\"

Context:
{context_block}""").strip()
        answer, _ = generate_answer(system_prompt, question,
                                    temperature=self.temperature, max_tokens=self.max_tokens)
        return self.postprocess(answer, question)

    def query(self, question: str) -> dict:
        t0 = time.time()
        effective_query, contexts = self.retrieve(question)
        answer  = self.generate(question, contexts)
        latency = round(time.time() - t0, 3)
        return {"question": question, "answer": answer, "contexts": contexts,
                "effective_query": effective_query, "latency_s": latency}

    def rerank_llm(self, query: str, contexts: list, top_k: int) -> list:
        scored = []
        for ctx in contexts:
            prompt = f"""Score the relevance of the following context to the question on a scale of 0 to 10.
Return ONLY a single integer between 0 and 10, nothing else.

Question: {query}

Context: {ctx}

Relevance score:"""
            try:
                response = get_llm_client(temperature=0).invoke(prompt).content.strip()
                score = float(re.findall(r'\d+', response)[0])
            except Exception:
                score = 0.0
            scored.append((ctx, score))
        return [ctx for ctx, _ in sorted(scored, key=lambda x: x[1], reverse=True)[:top_k]]


print("✅ RAGPipeline class defined")


✅ RAGPipeline class defined


---
## Cell 8 — RAGAS Evaluation

Runs RAGAS on a list of RAG results and returns a score dict.

**Metrics explained:**

| Metric | Type | Question it answers |
|---|---|---|
| `faithfulness` | LLM judge | Is the answer grounded in the retrieved context? |
| `context_recall` | Embedding | Does the retrieved context cover the ground truth? |
| `context_precision` | Embedding | How much of the retrieved context is actually useful? |
| `answer_correctness` | Hybrid | Does the answer match the ground truth? |
| `composite` | Average | Mean of all available scores |

**Concurrency note:** `RAGAS_CONCURRENCY=1` because Ollama queues requests serially.
High concurrency causes `TimeoutError` as jobs wait too long in the queue.
`RAGAS_TIMEOUT=300` covers the worst-case wait for all jobs to flush through.

In [13]:
def run_ragas(results: list, max_samples: int = None) -> dict:
    print("\n[ragas] Building evaluation dataset...")
    sample = results
    if max_samples and len(results) > max_samples:
        import random
        sample = random.sample(results, max_samples)
        print(f"  [ragas] Sampling {max_samples}/{len(results)} questions for scoring")

    dataset = HFDataset.from_dict({
        "user_input":         [r["question"]     for r in sample],
        "response":           [r["answer"]       for r in sample],
        "retrieved_contexts": [r["contexts"]     for r in sample],
        "reference":          [r["ground_truth"] for r in sample],
    })

    from ragas import EvaluationDataset
    from ragas.metrics import Faithfulness, ContextRecall, ContextPrecision, AnswerCorrectness
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from langchain_mistralai import ChatMistralAI
    from langchain_openai import OpenAIEmbeddings

    judge_raw = ChatMistralAI(
        model="mistral-small-latest",
        mistral_api_key=MISTRAL_API_KEY,
        temperature=0,
    )
    judge = LangchainLLMWrapper(judge_raw)

    # emb_raw = OpenAIEmbeddings(
    #     model=EMBEDDINGS_MODEL,
    #     base_url=EMBEDDINGS_BASE_URL,
    #     api_key=EMBEDDINGS_API_KEY or "not-needed",
    # )
    emb_raw = get_pipeline_embeddings()
    ragas_emb = LangchainEmbeddingsWrapper(emb_raw)

    metrics = [
        Faithfulness(llm=judge),
        ContextRecall(llm=judge),
        ContextPrecision(llm=judge),
        AnswerCorrectness(llm=judge),
    ]

    try:
        from ragas.run_config import RunConfig
        rc = RunConfig(timeout=120, max_retries=2, max_wait=30, max_concurrency=RAGAS_CONCURRENCY)
    except Exception:
        rc = None

    eval_dataset = EvaluationDataset.from_hf_dataset(dataset)

    eval_kwargs = dict(
        dataset=eval_dataset,
        metrics=metrics,
        embeddings=ragas_emb,
        raise_exceptions=False,
        show_progress=True,
    )
    if rc is not None:
        eval_kwargs["run_config"] = rc

    time.sleep(1) # brief pause before starting evaluation
    print("[ragas] Running evaluation...")
    score_obj = evaluate(**eval_kwargs)

    def _safe(key):
        v = score_obj[key]
        if v is None: return None
        if isinstance(v, (int, float)): return None if v != v else round(float(v), 4)
        if isinstance(v, list):
            vals = [float(x) for x in v if isinstance(x, (int, float)) and x == x]
            return round(sum(vals) / len(vals), 4) if vals else None
        try:
            f = float(v); return None if f != f else round(f, 4)
        except Exception: return None

    scores = {
        "faithfulness":       _safe("faithfulness"),
        "context_recall":     _safe("context_recall"),
        "context_precision":  _safe("context_precision"),
        "answer_correctness": _safe("answer_correctness"),
    }
    present = [v for v in scores.values() if isinstance(v, (int, float))]
    scores["composite"] = round(sum(present) / len(present), 4) if present else None

    print("\n[ragas] Results:")
    for k, v in scores.items():
        if isinstance(v, (int, float)):
            bar = "=" * int(v * 20) + "-" * (20 - int(v * 20))
            print(f"  {k:<25} [{bar}] {v:.4f}")
        else:
            print(f"  {k:<25} n/a  (judge LLM unavailable)")
    return scores

---
## Cell 9 — Experiment Runner

`run_experiment` ties everything together for a single configuration:
1. Builds a `RAGPipeline` from the config tuple
2. Ingests the corpus
3. Runs all eval questions **in parallel** using `ThreadPoolExecutor` (`EVAL_WORKERS` threads)
4. Computes p95 latency
5. Calls `run_ragas` to score the answers

The parallel step is important — without it, questions are answered one at a time even though the model server can handle concurrent requests.

In [14]:
def run_experiment(config: tuple, corpus_text: str, eval_rows: list) -> dict:
    # Support both old 12-field tuples (Exp 2-3) and new 18-field tuples (Exp 4+)
    if len(config) == 12:
        (chunk_size, overlap, top_k, use_reform, use_reranker, use_cross_encoder,
         use_multiquery, use_postprocessing, use_parent_doc, use_hybrid,
         use_llm_reranker, label) = config
        fetch_k = 20; lambda_mult = 0.7; temperature = 0.1
        max_tokens = 1024; similarity_threshold = 0.0; n_queries = 3
    else:
        (chunk_size, overlap, top_k, use_reform, use_reranker, use_cross_encoder,
         use_multiquery, use_postprocessing, use_parent_doc, use_hybrid, use_llm_reranker,
         fetch_k, lambda_mult, temperature, max_tokens, similarity_threshold, n_queries,
         label) = config

    print(f"\n{'='*70}")
    print(f"EXPERIMENT: {label}")
    print(f"  chunk={chunk_size}  overlap={overlap}  top_k={top_k}  fetch_k={fetch_k}  lambda={lambda_mult}")
    print(f"  reform={use_reform}  reranker={use_reranker}  cross_enc={use_cross_encoder}  llm_rerank={use_llm_reranker}")
    print(f"  multiquery={use_multiquery}(n={n_queries})  postproc={use_postprocessing}  parent_doc={use_parent_doc}  hybrid={use_hybrid}")
    print(f"  temp={temperature}  max_tokens={max_tokens}  sim_threshold={similarity_threshold}")
    print(f"{'='*70}")

    pipeline = RAGPipeline(
        chunk_size=chunk_size, chunk_overlap=overlap, top_k=top_k,
        use_reformulation=use_reform, use_reranker=use_reranker,
        use_cross_encoder=use_cross_encoder, use_multiquery=use_multiquery,
        use_postprocessing=use_postprocessing, use_parent_doc=use_parent_doc,
        use_hybrid=use_hybrid, use_llm_reranker=use_llm_reranker,
        fetch_k=fetch_k, lambda_mult=lambda_mult,
        temperature=temperature, max_tokens=max_tokens,
        similarity_threshold=similarity_threshold, n_queries=n_queries,
    )
    vs, chunks = get_or_build_vectorstore(
        corpus_text, chunk_size, overlap,
        use_parent_doc=use_parent_doc, use_hybrid=use_hybrid
    )
    pipeline.vectorstore = vs
    if use_hybrid:
        from langchain_community.retrievers import BM25Retriever
        pipeline.bm25   = BM25Retriever.from_documents(chunks)
        pipeline.bm25.k = top_k
    print(f"  index ready ({len(chunks)} chunks)")

    results = get_or_generate_answers(config, pipeline, eval_rows)

    latencies   = [r["latency_s"] for r in results if isinstance(r.get("latency_s"), (int, float))]
    p95_latency = float(np.percentile(latencies, 95)) if latencies else 0.0
    print(f"  p95 latency: {p95_latency:.3f}s")

    ragas_scores = run_ragas(results)

    return {
        "label":        label,
        "config":       {"chunk_size": chunk_size, "overlap": overlap, "top_k": top_k,
                         "reform": use_reform, "label": label,
                         "fetch_k": fetch_k, "lambda_mult": lambda_mult,
                         "temperature": temperature, "max_tokens": max_tokens,
                         "similarity_threshold": similarity_threshold, "n_queries": n_queries},
        "ragas_scores": ragas_scores,
        "results":      results,
        "p95_latency":  p95_latency,
    }


---
## Cell 10 — I/O Helpers

- `load_eval_csv` — reads `question` + `ground_truth` columns from your CSV
- `load_corpus` — loads a `.pdf` or `.txt` file into a single string
- `save_results` — writes a timestamped JSON (full) + CSV (summary) to `OUT_DIR`
- `_experiment_group` — extracts the group name prefix from an experiment label for grouped display

In [15]:
def load_eval_csv(path: str) -> list:
    rows = []
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            rows.append({"question": row["question"].strip(), "ground_truth": row["ground_truth"].strip()})
    print(f"[csv] Loaded {len(rows)} evaluation questions from {path}")
    return rows


def load_corpus(file_path: str) -> str:
    path = Path(file_path)
    if path.suffix.lower() == ".pdf":
        from langchain_community.document_loaders import PyPDFLoader
        return "\n".join(p.page_content for p in PyPDFLoader(str(path)).load())
    elif path.suffix.lower() == ".txt":
        return path.read_text(encoding="utf-8")
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}. Use .pdf or .txt")


def save_results(all_experiments: list, out_dir: str = "."):
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    os.makedirs(out_dir, exist_ok=True)
    json_path = os.path.join(out_dir, f"rag_eval_results_{ts}.json")
    with open(json_path, "w") as f:
        json.dump(all_experiments, f, indent=2, default=str)
    print(f"\nFull results  -> {json_path}")
    csv_path = os.path.join(out_dir, f"rag_eval_summary_{ts}.csv")
    rows = [{"experiment": e["label"], **e["config"], **e["ragas_scores"]} for e in all_experiments]
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    print(f"Summary CSV   -> {csv_path}")
    _print_summary_table(all_experiments)


def _experiment_group(label: str) -> str:
    m = re.match(r"^(.*?)(?:_top_k|top_k)", label)
    return m.group(1).strip("_") or label if m else label


def _print_summary_table(experiments: list, title: str = "FINAL RESULTS"):
    H = ["EXPERIMENT", "FAITHFUL", "RECALL", "PRECISION", "CORRECTNESS", "COMPOSITE"]
    w = 107
    def _f(v): return f"{v:>10.4f}" if isinstance(v, (int, float)) else f"{'n/a':>10}"
    print(f"\n{'─'*w}\n  {title}\n{'─'*w}")
    print(f"{H[0]:<28} {H[1]:>10} {H[2]:>8} {H[3]:>10} {H[4]:>12} {H[5]:>10}")
    print(f"{'─'*w}")
    for exp in experiments:
        s = exp["ragas_scores"]
        print(f"{exp['label']:<28}{_f(s.get('faithfulness'))}{_f(s.get('context_recall'))}{_f(s.get('context_precision'))}{_f(s.get('answer_correctness'))}{_f(s.get('composite'))}")
    print(f"{'─'*w}")
    scoreable = [e for e in experiments if isinstance(e["ragas_scores"].get("composite"), (int, float))]
    if scoreable:
        best = max(scoreable, key=lambda e: e["ragas_scores"]["composite"])
        print(f"\n🏆 Best config: {best['label']}  composite={best['ragas_scores']['composite']:.4f}")


print("✅ I/O helpers defined")

✅ I/O helpers defined


---
## Cell 12 — Load Data

Loads the evaluation CSV and the corpus. Run this before the experiment loop.
Re-run this cell if you want to swap in a different document without restarting.

In [16]:
eval_rows = load_eval_csv(EVAL_CSV)

if CORPUS_FILE:
    corpus_text = load_corpus(CORPUS_FILE)
    print(f"Using corpus: {CORPUS_FILE} ({len(corpus_text):,} characters)")
else:
    corpus_text = BUILTIN_CORPUS
    print(f"Using built-in ML corpus ({len(corpus_text):,} characters)")

[csv] Loaded 50 evaluation questions from eval_dataset.csv
Using built-in ML corpus (2,913 characters)


In [17]:
# ── Cell 12b — Corpus Cache (pickle FAISS index to skip re-embedding) ─────────
import pickle, hashlib

CACHE_DIR = Path(os.getenv("CACHE_DIR", ".cache"))
CACHE_DIR.mkdir(exist_ok=True)

def _corpus_cache_key(text: str, chunk_size: int, chunk_overlap: int) -> str:
    """Stable hash of corpus content + chunking params."""
    h = hashlib.sha256(f"{chunk_size}:{chunk_overlap}:{text}".encode()).hexdigest()[:16]
    return h

def get_or_build_vectorstore(corpus_text: str, chunk_size: int, chunk_overlap: int,
                              use_parent_doc: bool = False, use_hybrid: bool = False):
    """
    Return a cached FAISS vectorstore + doc list if a matching pickle exists,
    otherwise build it from scratch, save it, and return it.

    The cache key encodes corpus content + chunk_size + chunk_overlap.
    If you change the corpus or chunking params the cache is automatically bypassed.
    """
    key  = _corpus_cache_key(corpus_text, chunk_size, chunk_overlap)
    path = CACHE_DIR / f"vectorstore_{key}.pkl"

    if path.exists():
        print(f"  [cache] Loading vectorstore from {path}")
        with open(path, "rb") as f:
            data = pickle.load(f)
        vs     = FAISS.deserialize_from_bytes(data["faiss_bytes"], get_pipeline_embeddings(), allow_dangerous_deserialization=True)
        chunks = data["chunks"]
        print(f"  [cache] Loaded {len(chunks)} chunks — skipping embedding step ✅")
        return vs, chunks

    # Cache miss — build and save
    print(f"  [cache] No cache for key {key}, embedding corpus...")
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.create_documents([corpus_text], metadatas=[{"source": "corpus"}])
    if use_parent_doc:
        for i, chunk in enumerate(chunks):
            chunk.metadata["parent_id"]   = i // 3
            start = max(0, i - 1); end = min(len(chunks), i + 1)
            chunk.metadata["parent_text"] = " ".join(chunks[j].page_content for j in range(start, end))

    embeddings = get_pipeline_embeddings()
    vs = FAISS.from_documents(chunks, embeddings)
    faiss_bytes = vs.serialize_to_bytes()

    with open(path, "wb") as f:
        pickle.dump({"faiss_bytes": faiss_bytes, "chunks": chunks, "key": key,
                     "chunk_size": chunk_size, "chunk_overlap": chunk_overlap}, f)
    print(f"  [cache] Saved to {path} ({len(chunks)} chunks)")
    return vs, chunks

print(f"✅ Vectorstore cache helpers defined  (cache dir: {CACHE_DIR.resolve()})")

✅ Vectorstore cache helpers defined  (cache dir: C:\Users\Taktouk\Documents\Chatbot\Chatbot-Rag\rag_core\.cache)


In [18]:
# ── Cell 12c — Answer Cache (skip re-generation between eval runs) ──────────
import hashlib, json
from pathlib import Path

ANSWER_CACHE_DIR = Path(os.getenv("ANSWER_CACHE_DIR", ".cache/answers"))
ANSWER_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _answer_cache_key(config: tuple, eval_rows: list) -> str:
    """Stable hash of experiment config + questions (not ground truths — those don't affect generation)."""
    config_str = str(config)
    questions_str = str([r["question"] for r in eval_rows])
    h = hashlib.sha256(f"{config_str}:{questions_str}".encode()).hexdigest()[:16]
    return h

def get_or_generate_answers(config: tuple, pipeline, eval_rows: list) -> list:
    """
    Return cached answers if they exist, otherwise run the full RAG generation,
    save results to disk, and return them.
    """
    key  = _answer_cache_key(config, eval_rows)
    path = ANSWER_CACHE_DIR / f"answers_{key}.json"

    if path.exists():
        print(f"  [answer cache] Loading from {path}")
        with open(path) as f:
            results = json.load(f)
        print(f"  [answer cache] Loaded {len(results)} answers — skipping generation ✅")
        return results

    # Cache miss — generate
    print(f"  [answer cache] No cache for key {key}, generating answers...")

    def _run_one(row):
        result = pipeline.query(row["question"])
        result["ground_truth"] = row["ground_truth"]
        return result

    results = [None] * len(eval_rows)
    with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as executor:
        futures = {executor.submit(_run_one, row): idx for idx, row in enumerate(eval_rows)}
        for future in as_completed(futures):
            idx = futures[future]
            results[idx] = future.result()
            print(f"  [{idx+1}/{len(eval_rows)}] done — {results[idx]['latency_s']:.2f}s")

    with open(path, "w") as f:
        json.dump(results, f, indent=2, default=str)
    print(f"  [answer cache] Saved to {path}")
    return results

---
## Cell 13 — Run Experiments

Iterates through `EXPERIMENT_GRID`, running each config against all eval questions.
After each **group** of experiments finishes (e.g. all `semantic_reranker_top_k=*` variants),
an interim results table is printed so you don't have to wait until the very end.

A total wall-clock timer is printed at the end.

In [19]:
# import shutil
# shutil.rmtree(".cache", ignore_errors=True)
# print("Cache cleared ✅")

In [20]:
# _main_start = time.time()
# print(f"MISTRAL_API_KEY = '{os.getenv('MISTRAL_API_KEY', 'NOT FOUND')}'")
# print(f"langchain_mistralai installed: ", end="")
# # ── Cell 13a — Experiment 1: Chunk Size × Overlap ─────────────────────────
# # Fixed: top_k=5, no reranker, no hybrid
# # Grid: chunk[256, 512, 768, 1024, 1280, 1536, 1792, 2048] × overlap[0%, 15%, 25%]
# # Winner → best_chunk_size, best_chunk_overlap

# _exp1_grid = []
# for chunk_size in [256, 512, 768, 1024, 1280, 1536, 1792, 2048]:
#     for overlap_pct in [0.0, 0.15, 0.25]:
#         overlap = int(chunk_size * overlap_pct)
#         label = f"chunk={chunk_size}_overlap={overlap}"
#         _exp1_grid.append(
#             (chunk_size, overlap, 5, False, False, False, False, False, False, False, label)
#         )

# _exp1_results = []
# all_results = []
# for config in _exp1_grid:
#     result = run_experiment(config, corpus_text, eval_rows)
#     all_results.append(result)
#     _exp1_results.append(result)

# _print_summary_table(_exp1_results, title="EXPERIMENT 1: CHUNK SIZE × OVERLAP")

# # Pick winner by composite score
# _exp1_scoreable = [e for e in _exp1_results if isinstance(e["ragas_scores"].get("composite"), (int, float))]
# if _exp1_scoreable:
#     _exp1_winner = max(_exp1_scoreable, key=lambda e: e["ragas_scores"]["composite"])
#     best_chunk_size    = _exp1_winner["config"]["chunk_size"]
#     best_chunk_overlap = _exp1_winner["config"]["overlap"]
#     print(f"\n✅ Experiment 1 winner: chunk={best_chunk_size}, overlap={best_chunk_overlap}  "
#           f"(composite={_exp1_winner['ragas_scores']['composite']:.4f})")
# else:
#     # Fallback defaults if scoring failed
#     best_chunk_size, best_chunk_overlap = 512, 0
#     print("⚠️  No scoreable results — using fallback: chunk=512, overlap=0")

# elapsed = time.time() - _main_start
# mins, secs = divmod(elapsed, 60)

# print(f"\n⏱  Total time: {int(mins)}m {secs:.1f}s")

In [21]:
# # ── Cell 13b — Experiment 2: Top-K Sweep ─────────────────────────────────────
# # Baseline: chunk=512, overlap=128 (fixed)
# # Grid: top_k in [1, 2, 3, 4, 5, 6, 8, 10, 12, 15]
# # Winner → best_top_k
# _main_start = time.time()

# _exp2_chunk_size    = 512
# _exp2_chunk_overlap = 128

# _exp2_grid = []
# for top_k in [1, 2, 3, 4, 5, 6, 8, 10, 12, 15]:
#     label = f"chunk={_exp2_chunk_size}_overlap={_exp2_chunk_overlap}_top_k={top_k}"
#     _exp2_grid.append(
#         (_exp2_chunk_size, _exp2_chunk_overlap, top_k,
#          False, False, False, False, False, False, False, False, label)
#     )
# all_results = []
# _exp2_results = []
# for config in _exp2_grid:
#     result = run_experiment(config, corpus_text, eval_rows)
#     all_results.append(result)
#     _exp2_results.append(result)

# _print_summary_table(_exp2_results, title="EXPERIMENT 2: TOP-K (chunk=512, overlap=128)")

# # Pick winner by composite score
# _exp2_scoreable = [e for e in _exp2_results if isinstance(e["ragas_scores"].get("composite"), (int, float))]
# if _exp2_scoreable:
#     _exp2_winner = max(_exp2_scoreable, key=lambda e: e["ragas_scores"]["composite"])
#     best_top_k = _exp2_winner["config"]["top_k"]
#     print(f"\n✅ Experiment 2 winner: top_k={best_top_k}  "
#           f"(composite={_exp2_winner['ragas_scores']['composite']:.4f})")
# else:
#     best_top_k = 6
#     print("⚠️  No scoreable results — using fallback: top_k=6")

# elapsed = time.time() - _main_start
# mins, secs = divmod(elapsed, 60)
# print(f"\n⏱  Total time so far: {int(mins)}m {secs:.1f}s")

In [22]:
# # ── Cell 13c — Experiment 3: Retrieval Strategy ───────────────────────────────
# # Baseline: best_chunk_size, best_chunk_overlap, best_top_k (from Exp 1 & 2)
# # Grid: dense MMR, hybrid, multiquery, hybrid + multiquery

# _exp3_grid = [
#     (512, 128, 8, False, False, False, False, False, False, False, False, "retrieval=dense_mmr"),
#     (512, 128, 8, False, False, False, False, False, False, True, False,  "retrieval=hybrid"),
#     (512, 128, 8, False, False, False, True,  False, False, False, False, "retrieval=multiquery"),
#     (512, 128, 8, False, False, False, True,  False, False, True, False,  "retrieval=hybrid+multiquery"),
# ]
# all_results = []
# _exp3_results = []
# for config in _exp3_grid:
#     result = run_experiment(config, corpus_text, eval_rows)
#     all_results.append(result)
#     _exp3_results.append(result)

# _print_summary_table(_exp3_results, title="EXPERIMENT 3: RETRIEVAL STRATEGY")

# _exp3_scoreable = [e for e in _exp3_results if isinstance(e["ragas_scores"].get("composite"), (int, float))]
# if _exp3_scoreable:
#     _exp3_winner = max(_exp3_scoreable, key=lambda e: e["ragas_scores"]["composite"])
#     best_retrieval = _exp3_winner["config"]
#     best_use_hybrid     = best_retrieval["label"].find("hybrid")     >= 0
#     best_use_multiquery = best_retrieval["label"].find("multiquery") >= 0
#     print(f"\n✅ Experiment 3 winner: {_exp3_winner['label']}  "
#           f"(composite={_exp3_winner['ragas_scores']['composite']:.4f})")
# else:
#     best_use_hybrid, best_use_multiquery = False, False
#     print("⚠️  No scoreable results — using fallback: dense MMR, no multiquery")

# elapsed = time.time() - _main_start
# mins, secs = divmod(elapsed, 60)
# print(f"\n⏱  Total time so far: {int(mins)}m {secs:.1f}s")

In [23]:
# # ══════════════════════════════════════════════════════════════════════════════
# # Cell 13d — Experiments 4–12  (run overnight after Exp 2 & 3 are done)
# #
# # Picks up best_top_k, best_use_hybrid, best_use_multiquery from cells above.
# # Config tuple layout (18 fields + label):
# #   chunk_size, overlap, top_k,
# #   use_reform, use_reranker, use_cross_encoder,
# #   use_multiquery, use_postprocessing, use_parent_doc, use_hybrid, use_llm_reranker,
# #   fetch_k, lambda_mult, temperature, max_tokens, similarity_threshold, n_queries,
# #   label
# # ══════════════════════════════════════════════════════════════════════════════
# _main_start = time.time()
# # ── Rolling baseline — seeded from Exp 2 & 3 winners ─────────────────────────
# _B_CHUNK  = 512;              _B_OVERLAP = 128
# _B_TOPK   = 8     
# _B_HYBRID = True   if "best_use_hybrid"   in dir() else False
# # _B_MQ     = best_use_multiquery if "best_use_multiquery" in dir() else False

# _B_REFORM    = False;  _B_RERANKER = False;  _B_CROSS_ENC = False
# _B_LLM_RNK   = False;  _B_POSTPROC = False;  _B_PARENT    = False
# _B_FETCHK    = 20;     _B_LAMBDA   = 0.7
# _B_TEMP      = 0.1;    _B_MAXTOK   = 1024
# _B_SIMTHR    = 0.0;    _B_NQUERIES = 3

# print(f"Starting baseline: chunk={_B_CHUNK}, overlap={_B_OVERLAP}, top_k={_B_TOPK}, "
#       f"hybrid={_B_HYBRID}, multiquery={_B_MQ}")


# def _cfg(label, **ov):
#     """Build an 18-field config tuple from the rolling baseline + overrides."""
#     c = dict(chunk=_B_CHUNK, overlap=_B_OVERLAP, top_k=_B_TOPK,
#              reform=_B_REFORM, reranker=_B_RERANKER, cross_enc=_B_CROSS_ENC,
#              mq=_B_MQ, postproc=_B_POSTPROC, parent=_B_PARENT,
#              hybrid=_B_HYBRID, llm_rnk=_B_LLM_RNK,
#              fetch_k=_B_FETCHK, lm=_B_LAMBDA,
#              temp=_B_TEMP, maxtok=_B_MAXTOK,
#              simthr=_B_SIMTHR, nq=_B_NQUERIES)
#     c.update(ov)
#     return (c["chunk"], c["overlap"], c["top_k"],
#             c["reform"], c["reranker"], c["cross_enc"],
#             c["mq"], c["postproc"], c["parent"],
#             c["hybrid"], c["llm_rnk"],
#             c["fetch_k"], c["lm"],
#             c["temp"], c["maxtok"],
#             c["simthr"], c["nq"],
#             label)

# all_results = []
# def _run_grid(grid, title):
#     results = []
#     for config in grid:
#         r = run_experiment(config, corpus_text, eval_rows)
#         results.append(r)
#         all_results.append(r)
#     _print_summary_table(results, title=title)
#     scoreable = [e for e in results if isinstance(e["ragas_scores"].get("composite"), (int, float))]
#     winner = max(scoreable, key=lambda e: e["ragas_scores"]["composite"]) if scoreable else None
#     if winner:
#         print(f"\n✅ Winner: {winner['label']}  composite={winner['ragas_scores']['composite']:.4f}")
#     else:
#         print("\n⚠️  No scoreable results — baseline unchanged")
#     elapsed = time.time() - _main_start
#     mins, secs = divmod(elapsed, 60)
#     print(f"⏱  Elapsed: {int(mins)}m {secs:.1f}s")
#     return results, winner


# # ════════════════════════════════════════════════════════════════════════════
# # Exp 4 — Reranker Strategy
# # ════════════════════════════════════════════════════════════════════════════
# _exp4_results, _exp4_winner = _run_grid([
#     _cfg("reranker=none"),
#     _cfg("reranker=semantic",      reranker=True),
#     _cfg("reranker=cross_encoder", cross_enc=True),
#     _cfg("reranker=llm",           llm_rnk=True),
# ], "EXP 4 — RERANKER STRATEGY")

# if _exp4_winner:
#     _B_RERANKER  = "semantic"      in _exp4_winner["label"]
#     _B_CROSS_ENC = "cross_encoder" in _exp4_winner["label"]
#     _B_LLM_RNK   = "llm"           in _exp4_winner["label"]


# # ════════════════════════════════════════════════════════════════════════════
# # Exp 5 — Similarity Threshold  (precision-first fix)
# # ════════════════════════════════════════════════════════════════════════════
# _exp5_results, _exp5_winner = _run_grid([
#     _cfg(f"sim_thr={t}", simthr=t) for t in [0.0, 0.3, 0.5, 0.7]
# ], "EXP 5 — SIMILARITY THRESHOLD")

# if _exp5_winner:
#     _B_SIMTHR = _exp5_winner["config"]["similarity_threshold"]


# # ════════════════════════════════════════════════════════════════════════════
# # Exp 6 — Temperature
# # ════════════════════════════════════════════════════════════════════════════
# _exp6_results, _exp6_winner = _run_grid([
#     _cfg(f"temp={t}", temp=t) for t in [0.0, 0.3, 0.7, 1.0]
# ], "EXP 6 — TEMPERATURE")

# if _exp6_winner:
#     _B_TEMP = _exp6_winner["config"]["temperature"]


# # ════════════════════════════════════════════════════════════════════════════
# # Exp 7 — fetch_k × lambda_mult  (MMR diversity)
# # ════════════════════════════════════════════════════════════════════════════
# _exp7_results, _exp7_winner = _run_grid([
#     _cfg(f"fetch_k={fk}_lambda={lm}", fetch_k=fk, lm=lm)
#     for fk in [10, 20, 50]
#     for lm in [0.3, 0.5, 0.7, 1.0]
# ], "EXP 7 — FETCH-K × LAMBDA_MULT")

# if _exp7_winner:
#     _B_FETCHK = _exp7_winner["config"]["fetch_k"]
#     _B_LAMBDA = _exp7_winner["config"]["lambda_mult"]


# # ════════════════════════════════════════════════════════════════════════════
# # Exp 8 — max_tokens
# # ════════════════════════════════════════════════════════════════════════════
# _exp8_results, _exp8_winner = _run_grid([
#     _cfg(f"max_tokens={m}", maxtok=m) for m in [256, 512, 1024, 2048]
# ], "EXP 8 — MAX TOKENS")

# if _exp8_winner:
#     _B_MAXTOK = _exp8_winner["config"]["max_tokens"]


# # ════════════════════════════════════════════════════════════════════════════
# # Exp 9 — Query Reformulation
# # ════════════════════════════════════════════════════════════════════════════
# _exp9_results, _exp9_winner = _run_grid([
#     _cfg("reform=off"),
#     _cfg("reform=on", reform=True),
# ], "EXP 9 — QUERY REFORMULATION")

# if _exp9_winner:
#     _B_REFORM = "on" in _exp9_winner["label"]


# # ════════════════════════════════════════════════════════════════════════════
# # Exp 10 — Postprocessing × Parent Doc
# # ════════════════════════════════════════════════════════════════════════════
# _exp10_results, _exp10_winner = _run_grid([
#     _cfg("postproc=off_parent=off"),
#     _cfg("postproc=on_parent=off",  postproc=True),
#     _cfg("postproc=off_parent=on",  parent=True),
#     _cfg("postproc=on_parent=on",   postproc=True, parent=True),
# ], "EXP 10 — POSTPROCESSING × PARENT DOC")

# if _exp10_winner:
#     _B_POSTPROC = "postproc=on" in _exp10_winner["label"]
#     _B_PARENT   = "parent=on"   in _exp10_winner["label"]


# # ════════════════════════════════════════════════════════════════════════════
# # Exp 11 — n_queries  (multiquery variant count)
# # ════════════════════════════════════════════════════════════════════════════
# _exp11_results, _exp11_winner = _run_grid([
#     _cfg(f"n_queries={n}", mq=True, nq=n) for n in [2, 3, 5]
# ], "EXP 11 — N_QUERIES")

# if _exp11_winner:
#     _B_NQUERIES = _exp11_winner["config"]["n_queries"]
#     _B_MQ       = True


# # ════════════════════════════════════════════════════════════════════════════
# # Exp 12 — Final Best Config
# # ════════════════════════════════════════════════════════════════════════════
# _exp12_results, _ = _run_grid([
#     _cfg("final_best",
#          reform=_B_REFORM,   reranker=_B_RERANKER, cross_enc=_B_CROSS_ENC,
#          mq=_B_MQ,           postproc=_B_POSTPROC, parent=_B_PARENT,
#          hybrid=_B_HYBRID,   llm_rnk=_B_LLM_RNK,
#          fetch_k=_B_FETCHK,  lm=_B_LAMBDA,
#          temp=_B_TEMP,       maxtok=_B_MAXTOK,
#          simthr=_B_SIMTHR,   nq=_B_NQUERIES)
# ], "EXP 12 — FINAL BEST CONFIG")


# # ════════════════════════════════════════════════════════════════════════════
# # Grand summary + save
# # ════════════════════════════════════════════════════════════════════════════
# _print_summary_table(all_results, title="GRAND SUMMARY — ALL EXPERIMENTS")
# save_results(all_results, out_dir=OUT_DIR)

# elapsed = time.time() - _main_start
# mins, secs = divmod(elapsed, 60)
# print(f"\n🏁  Total wall-clock time: {int(mins)}m {secs:.1f}s")


In [24]:
import time
import numpy as np

# ── Baseline (mirrors Cell 13d defaults) ─────────────────────────────────────
_main_start = time.time()

_B_CHUNK     = 512;   _B_OVERLAP  = 128
_B_TOPK      = 8
_B_REFORM    = False; _B_RERANKER = False; _B_CROSS_ENC = False
_B_MQ        = False; _B_POSTPROC = False; _B_PARENT    = False
_B_HYBRID    = True;  _B_LLM_RNK  = True
_B_FETCHK    = 20;    _B_LAMBDA   = 0.7
_B_TEMP      = 0;     _B_MAXTOK   = None   # None = unlimited (no max_tokens cap)
_B_SIMTHR    = 0.0;   _B_NQUERIES = 2

unlimited = None  # alias used in _cfg call below


def _cfg(label, **ov):
    c = dict(chunk=_B_CHUNK, overlap=_B_OVERLAP, top_k=_B_TOPK,
             reform=_B_REFORM, reranker=_B_RERANKER, cross_enc=_B_CROSS_ENC,
             mq=_B_MQ, postproc=_B_POSTPROC, parent=_B_PARENT,
             hybrid=_B_HYBRID, llm_rnk=_B_LLM_RNK,
             fetch_k=_B_FETCHK, lm=_B_LAMBDA,
             temp=_B_TEMP, maxtok=_B_MAXTOK,
             simthr=_B_SIMTHR, nq=_B_NQUERIES)
    c.update(ov)
    return (c["chunk"], c["overlap"], c["top_k"],
            c["reform"], c["reranker"], c["cross_enc"],
            c["mq"], c["postproc"], c["parent"],
            c["hybrid"], c["llm_rnk"],
            c["fetch_k"], c["lm"],
            c["temp"], c["maxtok"],
            c["simthr"], c["nq"],
            label)


all_results = []

def _run_grid(grid, title):
    results = []
    for config in grid:
        r = run_experiment(config, corpus_text, eval_rows)
        results.append(r)
        all_results.append(r)
    _print_summary_table(results, title=title)
    scoreable = [e for e in results if isinstance(e["ragas_scores"].get("composite"), (int, float))]
    winner = max(scoreable, key=lambda e: e["ragas_scores"]["composite"]) if scoreable else None
    if winner:
        print(f"\n✅ Winner: {winner['label']}  composite={winner['ragas_scores']['composite']:.4f}")
    else:
        print("\n⚠️  No scoreable results")
    elapsed = time.time() - _main_start
    mins, secs = divmod(elapsed, 60)
    print(f"⏱  Elapsed: {int(mins)}m {secs:.1f}s")
    return results, winner


# ════════════════════════════════════════════════════════════════════════════
# Exp 13 — Final Targeted Config
# ════════════════════════════════════════════════════════════════════════════
_exp13_results, _exp13_winner = _run_grid([
    _cfg("final_targeted",
         reform=False,  reranker=False, cross_enc=False,
         mq=False,      postproc=False, parent=False,
         hybrid=True,   llm_rnk=True,
         fetch_k=20,    lm=0.7,
         temp=0,        maxtok=unlimited,
         simthr=0,      nq=2),
], "EXP 13 — FINAL TARGETED CONFIG")

save_results(all_results, out_dir=OUT_DIR)


EXPERIMENT: final_targeted
  chunk=512  overlap=128  top_k=8  fetch_k=20  lambda=0.7
  reform=False  reranker=False  cross_enc=False  llm_rerank=True
  multiquery=False(n=2)  postproc=False  parent_doc=False  hybrid=True
  temp=0  max_tokens=None  sim_threshold=0


C:\Users\Taktouk\AppData\Local\Temp\ipykernel_7208\365741887.py:28: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  _pipeline_embeddings = HuggingFaceEmbeddings(model_name=EMBEDDINGS_MODEL)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 269.41it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [embed] HuggingFace BAAI/bge-base-en-v1.5 ready
  [cache] Loading vectorstore from .cache\vectorstore_73e77ed1916b22df.pkl
  [cache] Loaded 9 chunks — skipping embedding step ✅
  index ready (9 chunks)
  [answer cache] No cache for key 0ec8fafb8ebb83c7, generating answers...
  [gen] mistral-small-latest (temp=0, max_tokens=1024) ready
  [gen] mistral-small-latest (temp=0, max_tokens=1024) ready
  [gen] mistral-small-latest (temp=0, max_tokens=1024) ready
  [gen] mistral-small-latest (temp=0, max_tokens=1024) ready
  [gen] mistral-small-latest (temp=0, max_tokens=None) ready
  [4/50] done — 7.44s
  [1/50] done — 9.09s
  [3/50] done — 9.19s
  [2/50] done — 9.91s
  [5/50] done — 6.14s
  [6/50] done — 5.33s
  [7/50] done — 5.63s
  [8/50] done — 5.84s
  [9/50] done — 6.51s
  [11/50] done — 5.91s
  [10/50] done — 6.96s
  [12/50] done — 10.96s
  [13/50] done — 11.82s
  [14/50] done — 12.63s
  [15/50] done — 13.11s
  [16/50] done — 7.79s
  [17/50] done — 3.99s
  [19/50] done — 4.98s
  [20/50

C:\Users\Taktouk\AppData\Local\Temp\ipykernel_7208\3145526038.py:17: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ContextRecall, ContextPrecision, AnswerCorrectness
C:\Users\Taktouk\AppData\Local\Temp\ipykernel_7208\3145526038.py:17: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import Faithfulness, ContextRecall, ContextPrecision, AnswerCorrectness
C:\Users\Taktouk\AppData\Local\Temp\ipykernel_7208\3145526038.py:17: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ra

[ragas] Running evaluation...


Evaluating: 100%|██████████| 200/200 [02:21<00:00,  1.41it/s]


[ragas] Results:
  faithfulness              [===-----------------] 0.1522
  context_recall            [--------------------] 0.0000
  context_precision         [--------------------] 0.0000
  answer_correctness        [==------------------] 0.1374
  composite                 [=-------------------] 0.0724

───────────────────────────────────────────────────────────────────────────────────────────────────────────
  EXP 13 — FINAL TARGETED CONFIG
───────────────────────────────────────────────────────────────────────────────────────────────────────────
EXPERIMENT                     FAITHFUL   RECALL  PRECISION  CORRECTNESS  COMPOSITE
───────────────────────────────────────────────────────────────────────────────────────────────────────────
final_targeted                  0.1522    0.0000    0.0000    0.1374    0.0724
───────────────────────────────────────────────────────────────────────────────────────────────────────────

🏆 Best config: final_targeted  composite=0.0724

✅ Winner: fin

---
## Cell 14 — Save Results

Writes two output files to `OUT_DIR` (default: current directory):
- `rag_eval_results_<timestamp>.json` — full output including every answer and retrieved context
- `rag_eval_summary_<timestamp>.csv` — one row per experiment with all RAGAS scores

Also prints the final comparison table and highlights the best-scoring config.

---
## Cell 15 — Inspect Individual Results *(optional)*

After the run you can dig into the raw results for any experiment.
Change `EXPERIMENT_LABEL` to the name of whichever experiment you want to inspect.

In [25]:
# EXPERIMENT_LABEL = "top_k=6"   # ← change to inspect a different experiment

# exp = next((e for e in all_results if e["label"] == EXPERIMENT_LABEL), None)
# if exp is None:
#     print(f"No experiment named '{EXPERIMENT_LABEL}'. Available: {[e['label'] for e in all_results]}")
# else:
#     print(f"Experiment : {exp['label']}")
#     print(f"p95 latency: {exp['p95_latency']:.3f}s")
#     print(f"RAGAS scores: {exp['ragas_scores']}\n")
#     for i, r in enumerate(exp["results"]):
#         print(f"--- Q{i+1} ({r['latency_s']:.2f}s) ---")
#         print(f"Question    : {r['question']}")
#         print(f"Answer      : {r['answer'][:300]}..." if len(r['answer']) > 300 else f"Answer      : {r['answer']}")
#         print(f"Ground truth: {r['ground_truth']}")
#         print(f"Contexts    : {len(r['contexts'])} chunks retrieved")
#         print()